<a href="https://colab.research.google.com/github/NayraSousa/SpikingJET/blob/public/SpikingJET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
class WeightFault:

    def __init__(self,
                 layer_name: str,
                 tensor_index: tuple,
                 bit: int,
                 value: int = None):
        self.layer_name = layer_name
        self.tensor_index = tensor_index
        self.bit = bit
        self.value = value

In [ ]:
import struct


def float32_bit_flip(golden_value: float,
                       bit: int) -> float:
    """
    Inject a bit-flip on a data represented as float32
    :param golden_value: the value to bit-flip
    :param bit: the bit where to perform the bit-flip
    :return: The value of the bit-flip on the golden value
    """
    float_list = []
    a = struct.pack('!f', golden_value)
    b = struct.pack('!I', int(2. ** bit))
    for ba, bb in zip(a, b):
        float_list.append(ba ^ bb)

    faulted_value = struct.unpack('!f', bytes(float_list))[0]

    return faulted_value

In [ ]:
import struct


class WeightFaultInjector:

    def __init__(self, network):

        self.network = network

        self.layer_name = None
        self.tensor_index = None
        self.bit = None

        self.golden_value = None


    def __inject_fault(self, layer_name, tensor_index, bit, value=None):
        self.layer_name = layer_name
        self.tensor_index = tensor_index
        self.bit = bit

        self.golden_value = float(self.network.state_dict()[self.layer_name][self.tensor_index])

        # If the value is not set, then we are doing a bit-flip
        if value is None:
            faulty_value = self.__float32_bit_flip()
        else:
            faulty_value = self.__float32_stuck_at(value)

        self.faulty_value = faulty_value

        self.network.state_dict()[self.layer_name][self.tensor_index] = faulty_value


    def __float32_bit_flip(self):
        """
        Inject a bit-flip on a data represented as float32
        :return: The value of the bit-flip on the golden value
        """
        float_list = []
        a = struct.pack('!f', self.golden_value)
        b = struct.pack('!I', int(2. ** self.bit))
        for ba, bb in zip(a, b):
            float_list.append(ba ^ bb)

        faulted_value = struct.unpack('!f', bytes(float_list))[0]

        return faulted_value

    def __float32_stuck_at(self,
                           value: int):
        """
        Inject a stuck-at fault on a data represented as float32
        :param value: the value to use as stuck-at value
        :return: The value of the bit-flip on the golden value
        """
        float_list = []
        a = struct.pack('!f', self.golden_value)
        b = struct.pack('!I', int(2. ** self.bit))
        for ba, bb in zip(a, b):
            if value == 1:
                float_list.append(ba | bb)
            else:
                float_list.append(ba & (255 - bb))

        faulted_value = struct.unpack('!f', bytes(float_list))[0]

        return faulted_value

    def restore_golden(self):
        """
        Restore the value of the faulted network weight to its golden value
        """
        if self.layer_name is None:
            print('CRITICAL ERROR: impossible to restore the golden value before setting a fault')
            quit()

        self.network.state_dict()[self.layer_name][self.tensor_index] = self.golden_value

    def inject_bit_flip(self,
                        layer_name: str,
                        tensor_index: tuple,
                        bit: int):
        """
        Inject a bit-flip in the specified layer at the tensor_index position for the specified bit
        :param layer_name: The name of the layer
        :param tensor_index: The index of the weight to fault inside the tensor
        :param bit: The bit where to inject the fault
        """
        self.__inject_fault(layer_name=layer_name,
                            tensor_index=tensor_index,
                            bit=bit)

    def inject_stuck_at(self,
                        layer_name: str,
                        tensor_index: tuple,
                        bit: int,
                        value: int):
        """
        Inject a stuck-at fault to the specified value in the specified layer at the tensor_index position for the
        specified bit
        :param layer_name: The name of the layer
        :param tensor_index: The index of the weight to fault inside the tensor
        :param bit: The bit where to inject the fault
        :param value: The stuck-at value to set
        """
        self.__inject_fault(layer_name=layer_name,
                            tensor_index=tensor_index,
                            bit=bit,
                            value=value)

In [ ]:
!pip install snntorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 5.1 MB/s eta 0:00:00


In [ ]:
import struct


class WeightFaultInjector:

    def __init__(self, network):

        self.network = network

        self.layer_name = None
        self.tensor_index = None
        self.bit = None

        self.golden_value = None


    def __inject_fault(self, layer_name, tensor_index, bit, value=None):
        self.layer_name = layer_name
        self.tensor_index = tensor_index
        self.bit = bit

        self.golden_value = float(self.network.state_dict()[self.layer_name][self.tensor_index])

        # If the value is not set, then we are doing a bit-flip
        if value is None:
            faulty_value = self.__float32_bit_flip()
        else:
            faulty_value = self.__float32_stuck_at(value)

        self.faulty_value = faulty_value

        self.network.state_dict()[self.layer_name][self.tensor_index] = faulty_value


    def __float32_bit_flip(self):
        """
        Inject a bit-flip on a data represented as float32
        :return: The value of the bit-flip on the golden value
        """
        float_list = []
        a = struct.pack('!f', self.golden_value)
        b = struct.pack('!I', int(2. ** self.bit))
        for ba, bb in zip(a, b):
            float_list.append(ba ^ bb)

        faulted_value = struct.unpack('!f', bytes(float_list))[0]

        return faulted_value

    def __float32_stuck_at(self,
                           value: int):
        """
        Inject a stuck-at fault on a data represented as float32
        :param value: the value to use as stuck-at value
        :return: The value of the bit-flip on the golden value
        """
        float_list = []
        a = struct.pack('!f', self.golden_value)
        b = struct.pack('!I', int(2. ** self.bit))
        for ba, bb in zip(a, b):
            if value == 1:
                float_list.append(ba | bb)
            else:
                float_list.append(ba & (255 - bb))

        faulted_value = struct.unpack('!f', bytes(float_list))[0]

        return faulted_value

    def restore_golden(self):
        """
        Restore the value of the faulted network weight to its golden value
        """
        if self.layer_name is None:
            print('CRITICAL ERROR: impossible to restore the golden value before setting a fault')
            quit()

        self.network.state_dict()[self.layer_name][self.tensor_index] = self.golden_value

    def inject_bit_flip(self,
                        layer_name: str,
                        tensor_index: tuple,
                        bit: int):
        """
        Inject a bit-flip in the specified layer at the tensor_index position for the specified bit
        :param layer_name: The name of the layer
        :param tensor_index: The index of the weight to fault inside the tensor
        :param bit: The bit where to inject the fault
        """
        self.__inject_fault(layer_name=layer_name,
                            tensor_index=tensor_index,
                            bit=bit)

    def inject_stuck_at(self,
                        layer_name: str,
                        tensor_index: tuple,
                        bit: int,
                        value: int):
        """
        Inject a stuck-at fault to the specified value in the specified layer at the tensor_index position for the
        specified bit
        :param layer_name: The name of the layer
        :param tensor_index: The index of the weight to fault inside the tensor
        :param bit: The bit where to inject the fault
        :param value: The stuck-at value to set
        """
        self.__inject_fault(layer_name=layer_name,
                            tensor_index=tensor_index,
                            bit=bit,
                            value=value)

In [ ]:
import os
import shutil
import time
import math
import random
from datetime import timedelta

import torch
from torch.nn import Module
from torch.utils.data import DataLoader

from tqdm import tqdm

class FaultInjectionManager:

    def __init__(self,
                 network: Module,
                 network_name: str,
                 device: torch.device,
                 loader: DataLoader):

        self.network = network
        self.network_name = network_name
        self.loader = loader
        self.device = device

        self.transient = False
        self.one = True

        self.inject = False
        self.current = None
        self.size = {}
        self.spike = {}
        self.faults = {}
        self.mask = {}
        self.num_step = 25


        # The clean output of the network after the first run
        self.clean_output_scores = list()
        self.clean_output_indices = list()

        # The weight fault injector
        self.weight_fault_injector = WeightFaultInjector(self.network)

        # The output dir
        self.label_output_dir = f'/content/drive/MyDrive/SpikingJET/output/{self.network_name}/pt/label/batch_size_{self.loader.batch_size}'
        self.clean_output_dir = f'/content/drive/MyDrive/SpikingJET/output/{self.network_name}/pt/clean/batch_size_{self.loader.batch_size}'
        self.faulty_output_dir = f'/content/drive/MyDrive/SpikingJET/output/{self.network_name}/pt/faulty/batch_size_{self.loader.batch_size}'

        # Create the output dir
        os.makedirs(self.label_output_dir, exist_ok=True)
        os.makedirs(self.clean_output_dir, exist_ok=True)
        os.makedirs(self.faulty_output_dir, exist_ok=True)

    def run_clean(self, fault_manager):
        """
        Run a clean inference of the network
        :return: A string containing the formatted time elapsed from the beginning to the end of the fault injection
        campaign
        """
        self.inject = False
        with torch.no_grad():

            # Start measuring the time elapsed
            start_time = time.time()

            # Cycle all the batches in the data loader
            pbar = tqdm(self.loader,
                        colour='green',
                        desc=f'Clean Run',
                        ncols=shutil.get_terminal_size().columns)

            for batch_id, batch in enumerate(pbar):
                #print(batch_id)
                data, label = batch
                #print(len(label)) total of 10000 images
                data = data.to(self.device)

                # Run inference on the current batch
                scores, indices = self.__run_inference_on_batch(data=data)

                # Save the output
                torch.save(scores, f'{self.clean_output_dir}/batch_{batch_id}.pt')
                torch.save(label, f'{self.label_output_dir}/batch_{batch_id}.pt')

                # Append the results to a list
                self.clean_output_scores.append(scores)
                self.clean_output_indices.append(indices)

        # Stop measuring the time
        elapsed = math.ceil(time.time() - start_time)

        return str(timedelta(seconds=elapsed))


    def run_faulty_campaign(self,
                            fault_list: list,
                            different_scores: bool = False) -> str:
        """
        Run a faulty injection campaign for the network
        :param fault_list: list of fault to inject
        :param different_scores: Default False. If True, compare the faulty scores with the clean scores, otherwise
        compare the top-1 indices
        :return: A string containing the formatted time elapsed from the beginning to the end of the fault injection
        campaign
        """

        total_different_predictions = 0
        total_predictions = 0

        self.inject = True
        with torch.no_grad():

            # Start measuring the time elapsed
            start_time = time.time()

            pbar = tqdm(self.loader,
                        total=len(self.loader) * len(fault_list),
                        colour='green',
                        desc=f'Fault Injection campaign',
                        ncols=shutil.get_terminal_size().columns * 2)
            # Cycle all the batches in the data loader
            for batch_id, batch in enumerate(self.loader):
                data, _ = batch
                data = data.to(self.device)

                # Inject all the faults in a single batch
                for fault_id, fault in enumerate(fault_list):

                    if fault.layer_name.split('.')[1] != 'potential' and fault.layer_name.split('.')[1] != 'spike':
                        # Inject faults in the weight
                        self.__inject_fault_on_weight(fault, fault_mode='stuck-at')
                        self.current = None
                    else:
                        self.current = [fault.layer_name.split('.')[0], fault.layer_name.split('.')[1], fault.tensor_index, fault.bit, random.randint(0,24)]

                    if (fault.layer_name.split('.')[1] != 'potential' and fault.layer_name.split('.')[1] != 'spike') or self.one == True:
                        # Run inference on the current batch
                        faulty_scores, faulty_indices = self.__run_inference_on_batch(data=data)

                        # Save the output
                        torch.save(faulty_scores, f'{self.faulty_output_dir}/fault_{fault_id}_batch_{batch_id}.pt')

                        # Measure the different predictions
                        if different_scores:
                            different_predictions = int(torch.ne(faulty_scores,
                                                                self.clean_output_scores[batch_id]).sum())
                        else:
                            different_predictions = int(torch.ne(torch.Tensor(faulty_indices),
                                                                torch.Tensor(self.clean_output_indices[batch_id])).sum())

                        # Measure the loss in accuracy
                        total_different_predictions += different_predictions
                        total_predictions += len(batch[0])
                        different_predictions_percentage = 100 * total_different_predictions / total_predictions
                        pbar.set_postfix({'Different': f'{different_predictions_percentage:.4f}%'})

                    if fault.layer_name.split('.')[1] != 'potential' and fault.layer_name.split('.')[1] != 'spike':
                        # Restore the golden value
                        self.weight_fault_injector.restore_golden()

                    # Update the progress bar
                    pbar.update(1)
        # Stop measuring the time
        elapsed = math.ceil(time.time() - start_time)

        return str(timedelta(seconds=elapsed))


    def __run_inference_on_batch(self,
                                 data: torch.Tensor):
        """
        Rim a fault injection on a single batch
        :param data: The input data from the batch
        :return: a tuple (scores, indices) where the scores are the vector score of each element in the batch and the
        indices are the argmax of the vector score
        """

        # Execute the network on the batch
        network_output = self.network(data)
        prediction = torch.topk(network_output, k=1)

        # Get the score and the indices of the predictions
        prediction_scores = network_output.cpu()
        prediction_indices = [int(fault) for fault in prediction.indices]

        return prediction_scores, prediction_indices

    def __inject_fault_on_weight(self,
                                 fault,
                                 fault_mode='stuck-at'):
        """
        Inject a fault in one of the weight of the network
        :param fault: The fault to inject
        :param fault_mode: Default 'stuck-at'. One of either 'stuck-at' or 'bit-flip'. Which kind of fault model to
        employ
        """

        if fault_mode == 'stuck-at':
            self.weight_fault_injector.inject_stuck_at(layer_name=f'{fault.layer_name}',
                                                       tensor_index=fault.tensor_index,
                                                       bit=fault.bit,
                                                       value=fault.value)

        elif fault_mode == 'bit-flip':
            self.weight_fault_injector.inject_bit_flip(layer_name=f'{fault.layer_name}',
                                                       tensor_index=fault.tensor_index,
                                                       bit=fault.bit,)
        else:
            print('FaultInjectionManager: Invalid fault mode')
            quit()

    def flatten_layer_dim(self, layer):
      '''
      For a given layer (Multi dimensional array) compute
      the dimension of the flatten array (Mono dimensional array)
      '''
      n = 1
      for dim in self.size[layer]:
        n = n*dim
      return n

    def compute_flatten_index(self, layer, index_arr):
      '''
      Given an index of the layer (Multi dimensional array) return
      the corresponding index of the flatten array
      '''
      index = 0
      for n in range(len(index_arr)):
        i = n+1
        tmp =  index_arr[-i]
        for y in range(n):
          i2 = y+1
          tmp = tmp *self.size[layer][-i2]
        index = index + tmp
      return index

    def compute_mask(self):
      '''
      Compute the mask to perform the xor injection
      '''
      for layer, faults in self.faults.items():
        if layer not in self.mask.keys():
          self.mask[layer] = torch.zeros(self.size[layer], dtype=torch.int32)
        for tensor_index, bit_index in faults:

          self.mask[layer][tensor_index] += 2**bit_index


    def injection_dict(self, pot, spike, layer, pot_or_spike, tensor_index, bit_index):
        '''
        inject faults using the dictionary
        '''
        if pot_or_spike == "potential":
          for i in range(self.size[layer][0]):
            golden_value = float(pot[(i,) + tensor_index])
            faulty_value = float32_bit_flip(golden_value=golden_value, bit=bit_index)
            pot[(i,) + tensor_index] = faulty_value
        else: #to be modified
          for i in range(self.size[layer][0]):
            golden_value = float(pot[(i,) + tensor_index])
            faulty_value = float32_bit_flip(golden_value=golden_value, bit=bit_index)
            pot[(i,) + tensor_index] = faulty_value

    def injection_xor(self, pot, layer):
      '''
        inject faults using the XOR mask
      '''
      pot_int = pot.view(torch.int)
      pot_int_incjeted = torch.bitwise_xor(pot_int, self.mask[layer])
      pot = pot_int_incjeted.view(torch.float)


    def injection(self, spike, pot, layer, curr_step):
      if self.inject:
        if self.current != None:
          if self.one == False:
            self.injection_xor(pot, layer)
          else:
            if self.current[0] == layer:
              if self.transient == False or (self.transient and curr_step == self.current[4]):

                self.injection_dict(pot, spike, layer, self.current[1], self.current[2], self.current[3])


      else:
        self.size[layer] = pot.shape
        self.spike[layer] = spike.shape

In [ ]:
import snntorch as snn
from snntorch import surrogate
from snntorch import functional as SF
from snntorch import utils
import torch.nn as nn
from torch.nn import functional as F
import torch

# Define Network
class CSNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.FaultInjector = None

        #Initialize parameters
        self.beta1 = torch.ones(14)*0.5
        self.beta2 = torch.ones(5)*0.5
        self.beta3 = torch.ones(11)*0.5

        self.threshold1 = torch.ones(14)
        self.threshold2 = torch.ones(5)
        self.threshold3 = torch.ones(11)

        self.gradient = surrogate.fast_sigmoid(slope=25)

        # Initialize layers
        self.conv1 = nn.Conv2d(2, 12, 5)
        self.lif1 = snn.Leaky(beta=self.beta1, threshold=self.threshold1, spike_grad=self.gradient, learn_beta=True, learn_threshold=True)
        self.conv2 = nn.Conv2d(12, 32, 5)
        self.lif2 = snn.Leaky(beta=self.beta2, threshold=self.threshold2, spike_grad=self.gradient, learn_beta=True, learn_threshold=True)
        self.fc1 = nn.Linear(800, 11)
        self.lif3 = snn.Leaky(beta=self.beta3, threshold=self.threshold3, spike_grad=self.gradient, learn_beta=True, learn_threshold=True, output=True)



    def forward(self, x):

        # Initialize hidden states and outputs at t=0
        mem_lif1 = self.lif1.init_leaky()
        mem_lif2 = self.lif2.init_leaky()
        mem_lif3 = self.lif3.init_leaky()

        nu_step = []
        mem_rec = []
        spk_rec = []

        x = nn.functional.interpolate(x, size=(2, 32, 32))
        curr_step = 0
        for step in range(x.shape[0]):

          cur1 = F.max_pool2d(self.conv1(x[step]), 2)
          spk1, mem_lif1 = self.lif1(cur1, mem_lif1)
          self.FaultInjector.injection(mem_lif1, spk1, 'lif1', curr_step)

          cur2 = F.max_pool2d(self.conv2(spk1), 2)
          spk2, mem_lif2 = self.lif2(cur2, mem_lif2)
          self.FaultInjector.injection(mem_lif2, spk2, 'lif2', curr_step)


          cur3 = self.fc1(spk2.flatten(1)) # batch x ....
          spk3, mem_lif3 = self.lif3(cur3, mem_lif3)
          self.FaultInjector.injection(mem_lif3, spk3, 'lif3', curr_step)

          spk_rec.append(spk3)
          curr_step+=1


        spk_rec =torch.stack(spk_rec)

        res_vec = spk_rec.sum(dim=0)

        res_vec = F.softmax(res_vec)
        return res_vec

In [ ]:
import snntorch as snn
from snntorch import surrogate
from snntorch import functional as SF
from snntorch import utils
import torch.nn as nn
from torch.nn import functional as F
import torch

# from FaultGenerators.utils import float32_bit_flip

class NMNIST(nn.Module):
    def __init__(self):
        super().__init__()

        self.FaultInjector = None

        self.beta1 = torch.ones(20)*0.5
        self.beta2 = torch.ones(10)*0.5

        self.threshold1 = torch.ones(20)
        self.threshold2 = torch.ones(10)

        self.spike_grad = surrogate.atan()

        # Initialize layers
        self.fc1 = nn.Linear(17*17*2, 20)
        self.lif1 = snn.Leaky(beta=self.beta1, threshold=self.threshold1, spike_grad=self.spike_grad, learn_beta=True, learn_threshold=True)
        self.fc2 = nn.Linear(20, 10)
        self.lif2 = snn.Leaky(beta=self.beta2, threshold=self.threshold2, spike_grad=self.spike_grad, learn_beta=True, learn_threshold=True, output=True)

    def forward(self, x):

        # Initialize hidden states at t=0
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()

        # Record the final layer
        spk2_rec = []
        x = nn.functional.interpolate(x, size=(2, 17, 17))
        x = x.view(x.shape[0], x.shape[1], -1)
        curr_step = 0
        for step in range(x.shape[0]):

            cur1 = self.fc1(x[step])
            spk1, mem1 = self.lif1(cur1, mem1)
            self.FaultInjector.injection(mem1, spk1, 'lif1', curr_step)

            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            self.FaultInjector.injection(mem2, spk2,'lif2', curr_step)

            spk2_rec.append(spk2)
            curr_step += 1
            # mem2_rec.append(mem2)
        spk2_rec = torch.stack(spk2_rec)
        res_vec = spk2_rec.sum(dim=0)
        res_vec = F.softmax(res_vec)

        return res_vec

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"]=":4096:8"


__all__ = ['ResNet', 'resnet20', 'resnet32', 'resnet44', 'resnet56', 'resnet110', 'resnet1202']


def _weights_init(m):
    classname = m.__class__.__name__
    #print(classname)
    if isinstance(m, nn.Linear) or isinstance(m, nn.Conv2d):
        init.kaiming_normal_(m.weight)


class LambdaLayer(nn.Module):
    def __init__(self, lambd):
        super(LambdaLayer, self).__init__()
        self.lambd = lambd

    def forward(self, x):
        return self.lambd(x)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, option='A'):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            if option == 'A':
                """
                For CIFAR10 ResNet paper uses option A.
                """
                self.shortcut = LambdaLayer(lambda x:
                                            F.pad(x[:, :, ::2, ::2], (0, 0, 0, 0, planes//4, planes//4), "constant", 0))
            elif option == 'B':
                self.shortcut = nn.Sequential(
                     nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                     nn.BatchNorm2d(self.expansion * planes)
                )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet, self).__init__()
        self.in_planes = 16

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.layer1 = self._make_layer(block, 16, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 32, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 64, num_blocks[2], stride=2)
        self.linear = nn.Linear(64, num_classes)

        self.apply(_weights_init)


    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion

        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = F.avg_pool2d(out, out.size()[3])
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


def resnet20():
    return ResNet(BasicBlock, [3, 3, 3])


def resnet32():
    return ResNet(BasicBlock, [5, 5, 5])


def resnet44():
    return ResNet(BasicBlock, [7, 7, 7])


def resnet56():
    return ResNet(BasicBlock, [9, 9, 9])


def resnet110():
    return ResNet(BasicBlock, [18, 18, 18])


def resnet1202():
    return ResNet(BasicBlock, [200, 200, 200])


def test(net):
    import numpy as np
    total_params = 0

    for x in filter(lambda p: p.requires_grad, net.parameters()):
        total_params += np.prod(x.data.numpy().shape)
    print("Total number of params", total_params)
    print("Total layers", len(list(filter(lambda p: p.requires_grad and len(p.data.size()) > 1, net.parameters()))))


if __name__ == "__main__":
    for net_name in __all__:
        if net_name.startswith('resnet'):
            print(net_name)
            test(globals()[net_name]())
            print()

resnet20
Total number of params 269722
Total layers 20

resnet32
Total number of params 464154
Total layers 32

resnet44
Total number of params 658586
Total layers 44

resnet56
Total number of params 853018
Total layers 56

resnet110
Total number of params 1727962
Total layers 110

resnet1202
Total number of params 19421274
Total layers 1202



In [ ]:
!pip install tonic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.5/108.5 kB 6.6 MB/s eta 0:00:00


In [ ]:
import tonic
import numpy as np

import snntorch as snn
from snntorch import surrogate

import torch
import torch.nn as nn
from torch.nn import functional as F


# Define Network
class SHD(nn.Module):
	def __init__(self, 	device):

		super().__init__()
		self.FaultInjector = None

		self.sigmoid_slope		= 100
		self.device = device
		self.tau_mem			= 10e-3
		self.tau_syn			= 5e-3

		#Network dimensions
		self.num_inputs = tonic.datasets.hsd.SHD.sensor_size[0]
		self.num_hidden = 200
		self.num_outputs = 20
		self.time_step = 0.001

		self.alpha1 = torch.ones(self.num_hidden)*(np.exp(-self.time_step/self.tau_syn))
		self.beta1 = torch.ones(self.num_hidden)*(np.exp(-self.time_step/self.tau_mem))
		self.threshold1 = torch.ones(self.num_hidden)

		self.alpha2 = torch.ones(self.num_outputs)*(np.exp(-self.time_step/self.tau_syn))
		self.beta2 = torch.ones(self.num_outputs)*(np.exp(-self.time_step/self.tau_mem))
		self.threshold2 = torch.ones(self.num_outputs)

		# Fast sigmoid surrogate gradient
		self.spike_grad = surrogate.fast_sigmoid(slope=self.sigmoid_slope)

		# Initialize layers
		self.fc1 = nn.Linear(self.num_inputs, self.num_hidden)
		self.fb1 = nn.Linear(self.num_hidden, self.num_hidden)

		self.lif1 = snn.Synaptic(alpha=self.alpha1, beta=self.beta1, threshold=self.threshold1,
						   learn_alpha=True, learn_beta=True, learn_threshold=True,
						   spike_grad = self.spike_grad)

		self.fc2 = nn.Linear(self.num_hidden, self.num_outputs)

		self.lif2 = snn.Synaptic(alpha=self.alpha2, beta=self.beta2, threshold=self.threshold2,
						   	learn_alpha=True, learn_beta=True, learn_threshold=True,
							spike_grad = self.spike_grad)

	def forward(self, input_spikes):
		input_spikes = input_spikes[:, :, 0, :]
		# Initialize hidden states at t=0
		syn1, mem1 = self.lif1.init_synaptic()
		syn2, mem2 = self.lif2.init_synaptic()

		# Record the final layer
		# spk2_rec = []
		mem2_rec = []

		input_spikes = input_spikes.float()
		spk1 = torch.zeros(self.num_hidden).to(self.device)

		curr_step = 0
		for step in range(input_spikes.shape[1]):

			cur1 = self.fc1(input_spikes[:, step, :]) + self.fb1(spk1)
			spk1, syn1, mem1 = self.lif1(cur1, syn1, mem1)
			self.FaultInjector.injection(mem1,spk1, 'lif1', curr_step)

			cur2 = self.fc2(spk1)
			spk2, syn2, mem2 = self.lif2(cur2, syn2, mem2)
			self.FaultInjector.injection(mem2, spk2, 'lif2', curr_step)

			# spk2_rec.append(spk2)
			mem2_rec.append(mem2)
			curr_step += 1

		# spk2_rec = torch.stack(spk2_rec, dim=0)
		# spk_stack = torch.sum(spk2_rec, dim=0)

		mem2_stack = torch.stack(mem2_rec, dim=0)
		mem_stack, _ = torch.max(mem2_stack,dim=0)

		# spk_stack = F.softmax(spk_stack,dim=1)
		mem_stack = F.softmax(mem_stack,dim=1)

		return mem_stack

In [ ]:
import os

from tqdm import tqdm

import torch
from torch.utils.data import TensorDataset
from torchvision import transforms
from torchvision.datasets import CIFAR10, ImageNet

import tonic
import tonic.transforms as transforms



def load_ImageNet_validation_set(batch_size,
                                 image_per_class=None,
                                 imagenet_folder='~/Datasets/ImageNet'):

    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])

    transform_validation = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        normalize,
    ])

    validation_dataset_folder = 'tmp'
    validation_dataset_path = f'{validation_dataset_folder}/imagenet_{image_per_class}.pt'

    try:
        if image_per_class is None:
            raise FileNotFoundError

        validation_dataset = torch.load(validation_dataset_path)
        print('Resized Imagenet loaded from disk')

    except FileNotFoundError:
        validation_dataset = ImageNet(root=imagenet_folder,
                                      split='val',
                                      transform=transform_validation)

        if image_per_class is not None:
            selected_validation_list = []
            image_class_counter = [0] * 1000
            for validation_image in tqdm(validation_dataset, desc='Resizing Imagenet Dataset', colour='Yellow'):
                if image_class_counter[validation_image[1]] < image_per_class:
                    selected_validation_list.append(validation_image)
                    image_class_counter[validation_image[1]] += 1
            validation_dataset = selected_validation_list

        os.makedirs(validation_dataset_folder, exist_ok=True)
        torch.save(validation_dataset, validation_dataset_path)

    # DataLoader is used to load the dataset
    # for training
    val_loader = torch.utils.data.DataLoader(dataset=validation_dataset,
                                             batch_size=batch_size,
                                             shuffle=False)
    print('Dataset loaded')

    return val_loader


def load_CIFAR10_datasets(train_batch_size=32, train_split=0.8, test_batch_size=1, test_image_per_class=None):

    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),                                       # Crop the image to 32x32
        transforms.RandomHorizontalFlip(),                                          # Data Augmentation
        transforms.ToTensor(),                                                      # Transform from image to pytorch tensor
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),   # Normalize the data (stability for training)
    ])
    transform_test = transforms.Compose([
        transforms.CenterCrop(32),                                                  # Crop the image to 32x32
        transforms.ToTensor(),                                                      # Transform from image to pytorch tensor
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),   # Normalize the data (stability for training)
    ])

    train_dataset = CIFAR10('weights/files/',
                            train=True,
                            transform=transform_train,
                            download=True)
    test_dataset = CIFAR10('weights/files/',
                           train=False,
                           transform=transform_test,
                           download=True)

    # If only a number of images is required per class, modify the test set
    if test_image_per_class is not None:
        image_tensors = list()
        label_tensors = list()
        image_class_counter = [0] * 10
        for test_image in test_dataset:
            if image_class_counter[test_image[1]] < test_image_per_class:
                image_tensors.append(test_image[0])
                label_tensors.append(test_image[1])
                image_class_counter[test_image[1]] += 1
        test_dataset = TensorDataset(torch.stack(image_tensors),
                                     torch.tensor(label_tensors))

    # Split the training set into training and validation
    train_split_length = int(len(train_dataset) * train_split)
    val_split_length = len(train_dataset) - train_split_length
    train_subset, val_subset = torch.utils.data.random_split(train_dataset,
                                                             lengths=[train_split_length, val_split_length],
                                                             generator=torch.Generator().manual_seed(1234))
    # DataLoader is used to load the dataset
    # for training
    train_loader = torch.utils.data.DataLoader(dataset=train_subset,
                                               batch_size=train_batch_size,
                                               shuffle=True)
    val_loader = torch.utils.data.DataLoader(dataset=val_subset,
                                             batch_size=train_batch_size,
                                             shuffle=True)

    test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                              batch_size=test_batch_size,
                                              shuffle=False)

    print('Dataset loaded')

    return train_loader, val_loader, test_loader

def load_DVSGesture_test_dataset(batch_size):
    ### Transforms
    size = tonic.datasets.DVSGesture.sensor_size

    # Denoise transform removes outlier events with inactive surrounding pixels for 10ms
    denoise_transform = transforms.Denoise(filter_time=10000)

    # ToFrame transform bins events into 25 clusters of frames
    frame_transform = transforms.ToFrame(sensor_size=size, n_time_bins=25)

    # Chain the transforms
    all_transform = transforms.Compose([denoise_transform, frame_transform])
    test_set = tonic.datasets.DVSGesture(save_to='/content/drive/MyDrive/SpikingJET/data', transform=all_transform, train=False)
    cached_testset = tonic.DiskCachedDataset(test_set, cache_path='/content/drive/MyDrive/SpikingJET/cache/dvsgesture/test')
    test_loader = torch.utils.data.DataLoader(cached_testset, batch_size=batch_size, shuffle=False, drop_last=True, collate_fn=tonic.collation.PadTensors(batch_first=False))

    return test_loader

def load_NMNIST_test_dataset(batch_size):
    sensor_size = tonic.datasets.NMNIST.sensor_size

    # Denoise removes isolated, one-off events
    # time_window
    frame_transform = transforms.Compose([transforms.Denoise(filter_time=10000),
                                        transforms.ToFrame(sensor_size=sensor_size,
                                                            n_time_bins=100)
                                        ])

    testset = tonic.datasets.NMNIST(save_to='/content/drive/MyDrive/SpikingJET/data', transform=frame_transform, train=False)

    # no augmentations for the testset
    cached_testset = tonic.DiskCachedDataset(testset, cache_path='/content/drive/MyDrive/SpikingJET/data/cache/nmnist/test')
    testloader = torch.utils.data.DataLoader(cached_testset, batch_size=batch_size, shuffle=False, drop_last=True,collate_fn=tonic.collation.PadTensors(batch_first=False))

    return testloader

def load_SHD_test_dataset(batch_size):
    transform = transforms.Compose(
        [
            transforms.ToFrame(
                sensor_size=tonic.datasets.hsd.SHD.sensor_size,
                n_time_bins=100,
            )
        ]
    )

    test_set 	= tonic.datasets.hsd.SHD(save_to='/content/drive/MyDrive/SpikingJET/data', train=False,transform = transform)

    test_loader 	= torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False, drop_last=True)
    return test_loader


def load_from_dict(network, device, path, function=None):
    if '.th' in path:
        # state_dict = torch.load(path, map_location=device)['state_dict']
        state_dict = torch.load(path, map_location=device, weights_only=True)
    else:
        state_dict = torch.load(path, map_location=device)

    if function is None:
        clean_state_dict = {key.replace('module.', ''): value for key, value in state_dict.items()}
    else:
        clean_state_dict = {key.replace('module.', ''): function(value) if not (('bn' in key) and ('weight' in key)) else value for key, value in state_dict.items()}

    network.load_state_dict(clean_state_dict, strict=False)

In [ ]:
import os
import shutil
import time
import math
import random
from datetime import timedelta

import torch
from torch.nn import Module
from torch.utils.data import DataLoader

from tqdm import tqdm



class FaultInjectionManager:

    def __init__(self,
                 network: Module,
                 network_name: str,
                 device: torch.device,
                 loader: DataLoader):

        self.network = network
        self.network_name = network_name
        self.loader = loader
        self.device = device

        self.transient = False
        self.one = True

        self.inject = False
        self.current = None
        self.size = {}
        self.spike = {}
        self.faults = {}
        self.mask = {}
        self.num_step = 25


        # The clean output of the network after the first run
        self.clean_output_scores = list()
        self.clean_output_indices = list()

        # The weight fault injector
        self.weight_fault_injector = WeightFaultInjector(self.network)

        # The output dir
        self.label_output_dir = f'/content/drive/MyDrive/SpikingJET/output/{self.network_name}/pt/label/batch_size_{self.loader.batch_size}'
        self.clean_output_dir = f'/content/drive/MyDrive/SpikingJET/output/{self.network_name}/pt/clean/batch_size_{self.loader.batch_size}'
        self.faulty_output_dir = f'/content/drive/MyDrive/SpikingJET/output/{self.network_name}/pt/faulty/batch_size_{self.loader.batch_size}'

        # Create the output dir
        os.makedirs(self.label_output_dir, exist_ok=True)
        os.makedirs(self.clean_output_dir, exist_ok=True)
        os.makedirs(self.faulty_output_dir, exist_ok=True)

    def run_clean(self, fault_manager):
        """
        Run a clean inference of the network
        :return: A string containing the formatted time elapsed from the beginning to the end of the fault injection
        campaign
        """
        self.inject = False
        with torch.no_grad():

            # Start measuring the time elapsed
            start_time = time.time()

            # Cycle all the batches in the data loader
            pbar = tqdm(self.loader,
                        colour='green',
                        desc=f'Clean Run',
                        ncols=shutil.get_terminal_size().columns)

            for batch_id, batch in enumerate(pbar):
                #print(batch_id)
                data, label = batch
                #print(len(label)) total of 10000 images
                data = data.to(self.device)

                # Run inference on the current batch
                scores, indices = self.__run_inference_on_batch(data=data)

                # Save the output
                torch.save(scores, f'{self.clean_output_dir}/batch_{batch_id}.pt')
                torch.save(label, f'{self.label_output_dir}/batch_{batch_id}.pt')

                # Append the results to a list
                self.clean_output_scores.append(scores)
                self.clean_output_indices.append(indices)

        # Stop measuring the time
        elapsed = math.ceil(time.time() - start_time)

        return str(timedelta(seconds=elapsed))


    def run_faulty_campaign(self,
                            fault_list: list,
                            different_scores: bool = False) -> str:
        """
        Run a faulty injection campaign for the network
        :param fault_list: list of fault to inject
        :param different_scores: Default False. If True, compare the faulty scores with the clean scores, otherwise
        compare the top-1 indices
        :return: A string containing the formatted time elapsed from the beginning to the end of the fault injection
        campaign
        """

        total_different_predictions = 0
        total_predictions = 0

        self.inject = True
        with torch.no_grad():

            # Start measuring the time elapsed
            start_time = time.time()

            pbar = tqdm(self.loader,
                        total=len(self.loader) * len(fault_list),
                        colour='green',
                        desc=f'Fault Injection campaign',
                        ncols=shutil.get_terminal_size().columns * 2)
            # Cycle all the batches in the data loader
            for batch_id, batch in enumerate(self.loader):
                data, _ = batch
                data = data.to(self.device)

                # Inject all the faults in a single batch
                for fault_id, fault in enumerate(fault_list):

                    if fault.layer_name.split('.')[1] != 'potential' and fault.layer_name.split('.')[1] != 'spike':
                        # Inject faults in the weight
                        self.__inject_fault_on_weight(fault, fault_mode='stuck-at')
                        self.current = None
                    else:
                        self.current = [fault.layer_name.split('.')[0], fault.layer_name.split('.')[1], fault.tensor_index, fault.bit, random.randint(0,24)]

                    if (fault.layer_name.split('.')[1] != 'potential' and fault.layer_name.split('.')[1] != 'spike') or self.one == True:
                        # Run inference on the current batch
                        faulty_scores, faulty_indices = self.__run_inference_on_batch(data=data)

                        # Save the output
                        torch.save(faulty_scores, f'{self.faulty_output_dir}/fault_{fault_id}_batch_{batch_id}.pt')

                        # Measure the different predictions
                        if different_scores:
                            different_predictions = int(torch.ne(faulty_scores,
                                                                self.clean_output_scores[batch_id]).sum())
                        else:
                            different_predictions = int(torch.ne(torch.Tensor(faulty_indices),
                                                                torch.Tensor(self.clean_output_indices[batch_id])).sum())

                        # Measure the loss in accuracy
                        total_different_predictions += different_predictions
                        total_predictions += len(batch[0])
                        different_predictions_percentage = 100 * total_different_predictions / total_predictions
                        pbar.set_postfix({'Different': f'{different_predictions_percentage:.4f}%'})

                    if fault.layer_name.split('.')[1] != 'potential' and fault.layer_name.split('.')[1] != 'spike':
                        # Restore the golden value
                        self.weight_fault_injector.restore_golden()

                    # Update the progress bar
                    pbar.update(1)
        # Stop measuring the time
        elapsed = math.ceil(time.time() - start_time)

        return str(timedelta(seconds=elapsed))


    def __run_inference_on_batch(self,
                                 data: torch.Tensor):
        """
        Rim a fault injection on a single batch
        :param data: The input data from the batch
        :return: a tuple (scores, indices) where the scores are the vector score of each element in the batch and the
        indices are the argmax of the vector score
        """

        # Execute the network on the batch
        network_output = self.network(data)
        prediction = torch.topk(network_output, k=1)

        # Get the score and the indices of the predictions
        prediction_scores = network_output.cpu()
        prediction_indices = [int(fault) for fault in prediction.indices]

        return prediction_scores, prediction_indices

    def __inject_fault_on_weight(self,
                                 fault,
                                 fault_mode='stuck-at'):
        """
        Inject a fault in one of the weight of the network
        :param fault: The fault to inject
        :param fault_mode: Default 'stuck-at'. One of either 'stuck-at' or 'bit-flip'. Which kind of fault model to
        employ
        """

        if fault_mode == 'stuck-at':
            self.weight_fault_injector.inject_stuck_at(layer_name=f'{fault.layer_name}',
                                                       tensor_index=fault.tensor_index,
                                                       bit=fault.bit,
                                                       value=fault.value)

        elif fault_mode == 'bit-flip':
            self.weight_fault_injector.inject_bit_flip(layer_name=f'{fault.layer_name}',
                                                       tensor_index=fault.tensor_index,
                                                       bit=fault.bit,)
        else:
            print('FaultInjectionManager: Invalid fault mode')
            quit()

    def flatten_layer_dim(self, layer):
      '''
      For a given layer (Multi dimensional array) compute
      the dimension of the flatten array (Mono dimensional array)
      '''
      n = 1
      for dim in self.size[layer]:
        n = n*dim
      return n

    def compute_flatten_index(self, layer, index_arr):
      '''
      Given an index of the layer (Multi dimensional array) return
      the corresponding index of the flatten array
      '''
      index = 0
      for n in range(len(index_arr)):
        i = n+1
        tmp =  index_arr[-i]
        for y in range(n):
          i2 = y+1
          tmp = tmp *self.size[layer][-i2]
        index = index + tmp
      return index

    def compute_mask(self):
      '''
      Compute the mask to perform the xor injection
      '''
      for layer, faults in self.faults.items():
        if layer not in self.mask.keys():
          self.mask[layer] = torch.zeros(self.size[layer], dtype=torch.int32)
        for tensor_index, bit_index in faults:

          self.mask[layer][tensor_index] += 2**bit_index


    def injection_dict(self, pot, spike, layer, pot_or_spike, tensor_index, bit_index):
        '''
        inject faults using the dictionary
        '''
        if pot_or_spike == "potential":
          for i in range(self.size[layer][0]):
            golden_value = float(pot[(i,) + tensor_index])
            faulty_value = float32_bit_flip(golden_value=golden_value, bit=bit_index)
            pot[(i,) + tensor_index] = faulty_value
        else: #to be modified
          for i in range(self.size[layer][0]):
            golden_value = float(pot[(i,) + tensor_index])
            faulty_value = float32_bit_flip(golden_value=golden_value, bit=bit_index)
            pot[(i,) + tensor_index] = faulty_value

    def injection_xor(self, pot, layer):
      '''
        inject faults using the XOR mask
      '''
      pot_int = pot.view(torch.int)
      pot_int_incjeted = torch.bitwise_xor(pot_int, self.mask[layer])
      pot = pot_int_incjeted.view(torch.float)


    def injection(self, spike, pot, layer, curr_step):
      if self.inject:
        if self.current != None:
          if self.one == False:
            self.injection_xor(pot, layer)
          else:
            if self.current[0] == layer:
              if self.transient == False or (self.transient and curr_step == self.current[4]):

                self.injection_dict(pot, spike, layer, self.current[1], self.current[2], self.current[3])


      else:
        self.size[layer] = pot.shape
        self.spike[layer] = spike.shape

In [ ]:
import os
import argparse

import torch

from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights, densenet121, DenseNet121_Weights
import matplotlib.pyplot as plt
import numpy as np
import torchvision.models as models
from scipy.stats import norm
from matplotlib.offsetbox import AnchoredText

class UnknownNetworkException(Exception):
    pass


def parse_args():
    """
    Parse the argument of the network
    :return: The parsed argument of the network
    """

    parser = argparse.ArgumentParser(description='Run a fault injection campaign',
                                     formatter_class=argparse.ArgumentDefaultsHelpFormatter)

    parser.add_argument('--forbid-cuda', action='store_true',
                        help='Completely disable the usage of CUDA. This command overrides any other gpu options.')
    parser.add_argument('--use-cuda', action='store_true',
                        help='Use the gpu if available.')
    parser.add_argument('--batch-size', '-b', type=int, default=64,
                        help='Test set batch size')
    parser.add_argument('--network-name', '-n', type=str,
                        required=True,
                        help='Target network',
                        choices=['ResNet20', 'ResNet32', 'ResNet44', 'ResNet56', 'ResNet110', 'ResNet1202',
                                 'DenseNet121', 'CSNN', 'NMNIST', 'SHD'])

    parsed_args = parser.parse_args()

    return parsed_args


def load_network(network_name: str,
                 device: torch.device) -> torch.nn.Module:
    """
    Load the network with the specified name
    :param network_name: The name of the network to load
    :param device: the device where to load the network
    :return: The loaded network
    """

    if 'ResNet' in network_name:
        if network_name == 'ResNet20':
            network_function = resnet20
        elif network_name == 'ResNet32':
            network_function = resnet32
        elif network_name == 'ResNet44':
            network_function = resnet44
        elif network_name == 'ResNet56':
            network_function = resnet56
        elif network_name == 'ResNet110':
            network_function = resnet110
        elif network_name == 'ResNet1202':
            network_function = resnet1202
        else:
            raise UnknownNetworkException(f'ERROR: unknown version of ResNet: {network_name}')

        # Instantiate the network
        network = network_function()

        # Load the weights
        network_path = f'/content/drive/MyDrive/SpikingJET/{network_name}.th'

        load_from_dict(network=network,
                       device=device,
                       path=network_path)
    elif 'DenseNet' in network_name:
        if network_name == 'DenseNet121':
            network = densenet121(weights=DenseNet121_Weights.DEFAULT)
        else:
            raise UnknownNetworkException(f'ERROR: unknown version of DenseNet: {network_name}')

    elif network_name == 'EfficientNet':
        network = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

    elif 'CSNN' in network_name:
        # Create instance of network
        network = CSNN()

        # Load the weights
        network_path =  f'/content/drive/MyDrive/SpikingJET/{network_name}.th'

        load_from_dict(network=network,
                       device=device,
                       path=network_path)
    elif 'NMNIST' in network_name:
        network = NMNIST()

        # Load the weights
        network_path =  f'/content/{network_name}.th'

        load_from_dict(network=network,
                device=device,
                path=network_path)
    elif 'SHD' in network_name:
        network = SHD(device)

        #Load the weights
        network_path = f'/content/{network_name}.th'

        load_from_dict(network=network,
        device=device,
        path=network_path)

    else:
        raise UnknownNetworkException(f'ERROR: unknown network: {network_name}')

    # Send network to device and set for inference
    network.to(device)
    network.eval()

    return network


def get_device(forbid_cuda: bool,
               use_cuda: bool) -> torch.device:
    """
    Get the device where to perform the fault injection
    :param forbid_cuda: Forbids the usage of cuda. Overrides use_cuda
    :param use_cuda: Whether to use the cuda device or the cpu
    :return: The device where to perform the fault injection
    """

    # Disable gpu if set
    if forbid_cuda:
        os.environ["CUDA_VISIBLE_DEVICES"] = ""
        device = 'cpu'
        if use_cuda:
            print('WARNING: cuda forcibly disabled even if set_cuda is set')
    # Otherwise, use the appropriate device
    else:
        if use_cuda:
            if torch.cuda.is_available():
                device = 'cuda'
            else:
                device = ''
                print('ERROR: cuda not available even if use-cuda is set')
                exit(-1)
        else:
            device = 'cpu'

    return torch.device(device)



#This function requires:
# -model: pytorch model
# -network: string of the model
# -dataset: name of the dataset

# def plot_distribution(model, network, dataset):
#     weights = [module.weight.flatten() for _, module in model.named_modules() if isinstance(module, torch.nn.Conv2d) or isinstance(module, torch.nn.BatchNorm2d) or isinstance(module, torch.nn.Linear)]
#     concatenated_weights = torch.cat(weights).tolist()
#     max_val=round(max(concatenated_weights),3)
#     min_val=round(min(concatenated_weights),3)
#     std_dev=round(np.std(concatenated_weights),3)
#     mean=round(np.mean(concatenated_weights),3)
#     # set up figure and axes
#     f, ax = plt.subplots(1,1)
#     #print(np.count_nonzero(concatenated_weights))
#     # Plot between -10 and 10 with .001 steps.
#     x_axis = np.arange(-5, 5, 0.001)
#     # Mean = 0, SD = 2.
#     plt.hist(concatenated_weights, bins=1000, density=True)
#     title_name="%s - %s" %(network, dataset)
#     plt.title(title_name)
#     plt.xlabel("x")
#     plt.ylabel("PDF(x)")
#     foldername="distributions_plots"
#     pathname="./%s/%s-%s-weights-distrib.png" %(foldername, network, dataset)
#     anchored_text = AnchoredText("Min=%s, Max=%s, Std Deviation=%s, Mean=%s" %(min_val, max_val, std_dev, mean), loc="upper right")
#     ax.add_artist(anchored_text)
    # plt.savefig(pathname)

In [ ]:
import itertools
import os
import csv
from tqdm import tqdm
import numpy as np
from ast import literal_eval as make_tuple

from typing import Type

from torch.nn import Module, Conv2d, Linear
import torch

import snntorch as snn


class FaultListGenerator:

    def __init__(self,
                 network: Module,
                 network_name: str,
                 device: torch.device,
                 module_classes: Type[Module] = (Conv2d, snn.Leaky, Linear, snn.Synaptic)):

        self.network = network
        self.network_name = network_name

        self.device = device

        # Name of the injectable layers
        injectable_layer_names = [name.split('.')[0] for name, module in self.network.named_modules()
                                  if isinstance(module, module_classes)]

        # List of the shape of all the layers that contain weight
        self.net_layer_shape = {name: param.shape for name, param in self.network.named_parameters()
                                if name.split('.')[0] in injectable_layer_names}
        #print(self.net_layer_shape)

        # List of the injectable params
        self.net_layer_params = {name: param for name, param in self.network.named_parameters()
                                 if name.split('.')[0] in injectable_layer_names}
        #print(self.net_layer_params)

        #print(self.network.state_dict())
    @staticmethod
    def __compute_date_n(N: int,
                         p: float = 0.5,
                         e: float = 0.01,
                         t: float = 2.58):
        """
        Compute the number of faults to inject according to the DATE09 formula
        :param N: The total number of parameters
        :param p: Default 0.5. The probability of a fault
        :param e: Default 0.01. The desired error rate
        :param t: Default 2.58. The desired confidence level
        :return: the number of fault to inject
        """
        return N / (1 + e ** 2 * (N - 1) / (t ** 2 * p * (1 - p)))

    def get_fault_list(self,
                                load_fault_list=False,
                                save_fault_list=True,
                                seed=51196,
                                p=0.5,
                                e=0.01,
                                t=2.58,
                                FaultInjectionManager=None):
        """
        Generate a fault list for the potential and beta of leaky layers and  according to the DATE09 formula
        :param load_fault_list: Default False. Try to load an existing fault list if it exists, otherwise generate it
        :param save_fault_list: Default True. Whether to save the fault list to file
        :param seed: Default 51195. The seed of the fault list
        :param p: Default 0.5. The probability of a fault
        :param e: Default 0.01. The desired error rate
        :param t: Default 2.58. The desired confidence level
        :return: The fault list
        """

        for layer, size in FaultInjectionManager.size.items():

            size_no_batch = size[1:] #removing the batch correlated dimension
            self.net_layer_shape[layer+".potential"] = size_no_batch

        for layer, size in FaultInjectionManager.spike.items():

            size_no_batch = size[1:] #removing the batch correlated dimension
            self.net_layer_shape[layer+".spike"] = size_no_batch

        # print(self.net_layer_shape)

        cwd = os.getcwd()
        fault_list_filename = f'/content/drive/MyDrive/SpikingJET/output/fault_list/{self.network_name}'

        try:
            if load_fault_list:
                with open(f'{fault_list_filename}/{seed}_fault_list.csv', newline='') as f_list:
                    reader = csv.reader(f_list)

                    fault_list = list(reader)[1:]

                    fault_list = [WeightFault(layer_name=fault[1],
                                              tensor_index=make_tuple(fault[2]),
                                              bit=int(fault[-1])) for fault in fault_list]

                print('Fault list loaded from file')

            # If you don't have to load the fault list raise the Exception and force the generation
            else:
                raise FileNotFoundError

        except FileNotFoundError:

            exhaustive_fault_list = []
            pbar = tqdm(self.net_layer_shape.items(), desc='Generating fault list', colour='green')
            for layer_name, layer_shape in pbar:
                # Add all the possible faults to the fault list
                k = np.arange(layer_shape[0])
                dim1 = np.arange(layer_shape[1]) if len(layer_shape) > 1 else [None]
                dim2 = np.arange(layer_shape[2]) if len(layer_shape) > 2 else [None]
                dim3 = np.arange(layer_shape[3]) if len(layer_shape) > 3 else [None]
                bits = np.arange(0, 32)

                exhaustive_fault_list = exhaustive_fault_list + list(
                    itertools.product(*[[layer_name], k, dim1, dim2, dim3, bits]))
                print(layer_name)
                print(len(list(itertools.product(*[[layer_name], k, dim1, dim2, dim3, bits]))))
            random_generator = np.random.default_rng(seed=seed)
            n = self.__compute_date_n(N=len(exhaustive_fault_list),
                                      p=p,
                                      e=e,
                                      t=t)

            fault_list = random_generator.choice(exhaustive_fault_list, int(n), replace=False)
            del exhaustive_fault_list
            fault_list = [WeightFault(layer_name=fault[0],
                                      tensor_index=tuple([int(i) for i in fault[1:-1]if i is not None]),
                                      bit=int(fault[-1])) for fault in fault_list]

            if save_fault_list:
                os.makedirs(fault_list_filename, exist_ok=True)
                with open(f'{fault_list_filename}/{seed}_fault_list.csv', 'w', newline='') as f_list:
                    writer_fault = csv.writer(f_list)

                    writer_fault.writerow(['Injection',
                                           'Layer',
                                           'TensorIndex',
                                           'Bit'])

                    golden_value_list = list()
                    faulty_value_list = list()
                    for index, fault in enumerate(fault_list):

                        layer = fault.layer_name.split('.')[0]
                        location = fault.layer_name.split('.')[1]
                        if location != 'potential' and location != 'spike': #this controll checks if the fault is on potentian or spike, in this case jump this code cause fault value is not available

                            # Get the golden value
                            golden_value = float(self.net_layer_params[fault.layer_name][fault.tensor_index])
                            # Get the faulty value
                            faulty_value = float32_bit_flip(golden_value=golden_value, bit=fault.bit)
                            # Append results to list
                            golden_value_list.append(golden_value)
                            faulty_value_list.append(faulty_value)

                        else:
                            golden_value_list.append(0)
                            faulty_value_list.append(0)
                            if layer not in FaultInjectionManager.faults.keys():
                                FaultInjectionManager.faults[layer] = []
                            FaultInjectionManager.faults[layer].append([fault.tensor_index, fault.bit])
                        # Write fault list to csv
                        writer_fault.writerow([index, fault.layer_name, fault.tensor_index, fault.bit])

                    # Write fault information to numpy
                    np.savez(f'{fault_list_filename}/{seed}_weights', golden=golden_value_list, faulty=faulty_value_list)

            #self.network.compute_mask()
            print('Fault List Generated')

        return fault_list

In [ ]:
def main():

    # Set deterministic algorithms
    torch.use_deterministic_algorithms(mode=True)

    # Select the device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device {device}')
    batch_size = 16
    network_name="CSNN"

    # Load the network
    network = load_network(network_name=network_name,
                           device=device)

    network.eval()

    #This function plots the weights' distribution of the CNN
    #plot_distribution(model=network, network=.network_name, dataset="dataset_name") # Plots CNN, BatchNorm2d and Linear layers weight distribution for visualization purposes

    # Load the dataset
    if 'ResNet' in network_name:
        _, _, loader = load_CIFAR10_datasets(test_batch_size=batch_size)
    elif 'CSNN' in network_name:
        loader = load_DVSGesture_test_dataset(batch_size=batch_size)
    elif 'NMNIST' in network_name:
        loader = load_NMNIST_test_dataset(batch_size=batch_size)
    elif 'SHD' in network_name:
        loader = load_SHD_test_dataset(batch_size=batch_size)
    else:
        loader = load_ImageNet_validation_set(batch_size=batch_size,
                                              image_per_class=1)



    # Execute the fault injection campaign with the smart network
    fault_injection_executor = FaultInjectionManager(network=network,
                                                     network_name=network_name,
                                                     device=device,
                                                     loader=loader)

    network.FaultInjector = fault_injection_executor

    fault_manager = FaultListGenerator(network=network,
                                       network_name=network_name,
                                       device=device)

    #This function runs clean inferences on the golden dataset
    fault_injection_executor.run_clean(fault_manager)


    # Generate fault list
    fault_list = fault_manager.get_fault_list(load_fault_list=False,
                                                     save_fault_list=True,
                                                     FaultInjectionManager = fault_injection_executor)

    #This function runs fault injections
    fault_injection_executor.run_faulty_campaign(fault_list=fault_list)


if __name__ == '__main__':
    main()

Using device cuda


Clean Run:   0%|                                                              | 0/7 [00:00<?, ?it/s]<ipython-input-8-55fd46f6723d>:73: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  res_vec = F.softmax(res_vec)
Generating fault list:  28%|██▊       | 5/18 [00:00<00:00, 49.03it/s]

conv1.weight
19200
conv1.bias
384
lif1.threshold
448
lif1.beta
448
conv2.weight
307200
conv2.bias
1024
lif2.threshold
160
lif2.beta
160
fc1.weight


Generating fault list:  83%|████████▎ | 15/18 [00:00<00:00, 40.94it/s]

281600
fc1.bias
352
lif3.threshold
352
lif3.beta
352
lif1.potential
75264
lif2.potential
25600
lif3.potential
352
lif1.spike


Generating fault list: 100%|██████████| 18/18 [00:00<00:00, 36.28it/s]


75264
lif2.spike
25600
lif3.spike
352
Fault List Generated


Fault Injection campaign: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 114149/114149 [2:41:02<00:00, 11.81it/s, Different=0.5915%]
